# Mock cabinet generator

Makes a test file of cabinets in the format of `Dataset/mock_cabinets.csv`, for the clustering notebooks, the
partition pipeline and the dashboard's **New run** tab. Change the settings below and run the whole notebook.

Cabinets are placed around real town centres in the chosen region, plus a share spread more widely between the
towns. Each cabinet gets a ticket count for the period, and **demand** = tickets / days. Each cabinet also gets a
**partition**: the team whose work it is. The data is made up and only meant for testing.

## Settings
Everything you can change is in the next cell. The same settings and `SEED` always give the same file;
`Dataset/mock_cabinets.csv` was made with the values below.

In [ ]:
N_CABINETS = 400               # how many cabinets to make
REGION = "Klang Valley"       # Klang Valley, Penang, Johor Bahru, Ipoh, Kota Kinabalu, Kuching, or "Custom"
AREA_KM = 45                  # cabinets stay inside a square this wide (km) around the region centre
N_TOWNS = 6                   # use the N biggest towns of the region (the list is ordered by size)
TOWN_RADIUS_KM = 3            # how far cabinets usually spread around a town centre
SCATTERED_SHARE = 0.25        # share of cabinets spread more widely between the towns (0 to 1)
SCATTER_RADIUS_KM = 12        # how far those scattered cabinets usually spread

TOTAL_DAYS = 90               # length of the ticket period (days)
MEDIAN_TICKETS = 54           # a typical cabinet's tickets over the period
TICKET_SPREAD = 0.8           # how uneven tickets are between cabinets (0 = all equal, 1+ = a few very busy ones)
MAX_DEMAND_PER_CABINET = 9    # no cabinet gets more than this many tickets/day

# Partitions: the area is cut into strips from west to east, one per partition, sized by these shares
# ({} = no partition column)
PARTITION_SHARES = {"TROS_1": 0.25, "TRIS_1": 0.5, "TROS_2": 0.25}
SHARED_SHARE = 0.02           # share of TRIS cabinets that are also work for the TROS partition next to them

# Rules for the check and the summary only (the ALL row of parameters.csv)
PEOPLE_PER_CLUSTER = 6
DEMAND_PER_PERSON = 6
SMALLEST_MIN = 4              # tickets/day of the smallest team at its minimum (1 person x 4)

SEED = 67
OUTPUT_NAME = "mock_cabinets.csv"   # file name in Dataset/; None = generated_<region>_<n>cab_seed<seed>.csv

# Your own towns, used when REGION = "Custom": (name, latitude, longitude, relative size)
CUSTOM_TOWNS = [
    ("Town A", 3.139, 101.687, 3),
    ("Town B", 3.073, 101.518, 2),
    ("Town C", 2.993, 101.787, 1),
]

In [ ]:
import math
import sys
from pathlib import Path

import folium
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# The notebook may be run from the project root or from tools/
ROOT = Path.cwd().parent if Path.cwd().name == "tools" else Path.cwd()
DATA_DIR = ROOT / "Dataset"
CAP = PEOPLE_PER_CLUSTER * DEMAND_PER_PERSON
rng = np.random.default_rng(SEED)
pd.set_option("display.width", 140)

## Region presets
Each region is a list of real towns: name, centre (latitude, longitude) and a rough relative size, so bigger towns
get more cabinets. The coordinates are approximate town centres. Regions on the coast (Penang, Johor Bahru,
Kota Kinabalu) can put a few cabinets in the sea; those have no road route, so the model can't place them.

In [ ]:
REGIONS = {
    "Klang Valley": [("Kuala Lumpur", 3.139, 101.687, 5), ("Petaling Jaya", 3.107, 101.607, 3),
                     ("Shah Alam", 3.073, 101.518, 3), ("Subang Jaya", 3.050, 101.585, 3),
                     ("Klang", 3.044, 101.445, 2), ("Kajang", 2.993, 101.787, 2), ("Ampang", 3.150, 101.762, 2),
                     ("Puchong", 3.025, 101.617, 2), ("Selayang", 3.250, 101.650, 1),
                     ("Putrajaya", 2.926, 101.696, 1)],
    "Penang": [("George Town", 5.414, 100.329, 4), ("Butterworth", 5.399, 100.364, 3),
               ("Bukit Mertajam", 5.363, 100.461, 2), ("Bayan Lepas", 5.296, 100.266, 2),
               ("Seberang Jaya", 5.397, 100.400, 1), ("Air Itam", 5.400, 100.278, 1),
               ("Nibong Tebal", 5.166, 100.478, 1)],
    "Johor Bahru": [("Johor Bahru", 1.466, 103.758, 4), ("Skudai", 1.535, 103.657, 2),
                    ("Johor Jaya", 1.540, 103.800, 2), ("Pasir Gudang", 1.470, 103.903, 2),
                    ("Iskandar Puteri", 1.425, 103.630, 1), ("Kulai", 1.660, 103.600, 1), ("Masai", 1.493, 103.878, 1)],
    "Ipoh": [("Ipoh", 4.597, 101.090, 4), ("Menglembu", 4.565, 101.040, 1), ("Chemor", 4.720, 101.115, 1),
             ("Tambun", 4.620, 101.150, 1), ("Simpang Pulai", 4.527, 101.123, 1), ("Batu Gajah", 4.469, 101.041, 1),
             ("Kampar", 4.311, 101.153, 1)],
    "Kota Kinabalu": [("Kota Kinabalu", 5.980, 116.073, 4), ("Penampang", 5.913, 116.111, 2),
                      ("Inanam", 6.016, 116.123, 2), ("Putatan", 5.882, 116.053, 1), ("Tuaran", 6.178, 116.232, 1),
                      ("Papar", 5.735, 115.932, 1)],
    "Kuching": [("Kuching", 1.553, 110.344, 4), ("Kota Samarahan", 1.459, 110.494, 2),
                ("Batu Kawa", 1.512, 110.290, 1), ("Matang", 1.580, 110.270, 1), ("Kota Padawan", 1.430, 110.330, 1),
                ("Bau", 1.417, 110.153, 1)],
}
all_towns = CUSTOM_TOWNS if REGION == "Custom" else REGIONS[REGION]
towns = pd.DataFrame(all_towns[:N_TOWNS], columns=["town", "lat", "long", "size"])
CENTRE = (towns["lat"].mean(), towns["long"].mean())
print(f"{REGION}: using {len(towns)} of {len(all_towns)} towns, centre {CENTRE[0]:.3f}, {CENTRE[1]:.3f}")
towns

## How cabinets are placed
- **Town cabinets**: pick a town (bigger towns more often) and place the cabinet a random distance from its centre,
  usually within `TOWN_RADIUS_KM`.
- **Scattered cabinets**: the same, but usually within the wider `SCATTER_RADIUS_KM`, so they fill the gaps between
  towns.
- Any cabinet that lands outside the `AREA_KM` square is placed again.

Distances are turned into latitude/longitude with the usual rule of thumb: 1 degree of latitude is about 111 km,
and a degree of longitude shrinks with the cosine of the latitude.

In [ ]:
def offset_km(lat, lon, dx_km, dy_km):
    # Move a point dx_km east and dy_km north.
    return lat + dy_km / 111.32, lon + dx_km / (111.32 * np.cos(np.radians(lat)))


def inside_area(lat, lon):
    # True for points inside the AREA_KM square around the region centre.
    dy = (lat - CENTRE[0]) * 111.32
    dx = (lon - CENTRE[1]) * 111.32 * np.cos(np.radians(CENTRE[0]))
    return (np.abs(dx) <= AREA_KM / 2) & (np.abs(dy) <= AREA_KM / 2)


def place_cabinets(n, spread_km):
    # n cabinets around the towns (picked by size), spread_km = typical distance from the town centre.
    lat = np.empty(n)
    lon = np.empty(n)
    town = np.empty(n, dtype=object)
    todo = np.arange(n)
    for _ in range(100):                      # redraw the ones outside the area
        pick = rng.choice(len(towns), size=len(todo), p=towns["size"] / towns["size"].sum())
        dx, dy = rng.normal(0, spread_km, size=(2, len(todo)))
        lat[todo], lon[todo] = offset_km(towns["lat"].to_numpy()[pick], towns["long"].to_numpy()[pick], dx, dy)
        town[todo] = towns["town"].to_numpy()[pick]
        todo = todo[~inside_area(lat[todo], lon[todo])]
        if len(todo) == 0:
            break
    if len(todo):
        raise ValueError(f"{len(todo)} cabinets could not be placed inside the area: make AREA_KM bigger "
                         "or the spread smaller.")
    return lat, lon, town


def make_tickets(n):
    # Tickets per cabinet over the period: most cabinets near MEDIAN_TICKETS, a few much busier (log-normal).
    tickets = np.round(MEDIAN_TICKETS * np.exp(rng.normal(0, TICKET_SPREAD, n)))
    return np.clip(tickets, 1, math.floor(MAX_DEMAND_PER_CABINET * TOTAL_DAYS)).astype(int)


def build_dataset():
    # The cabinet table, in the same columns and order as mock_cabinets.csv.
    n_scattered = int(round(N_CABINETS * SCATTERED_SHARE))
    lat1, lon1, town1 = place_cabinets(N_CABINETS - n_scattered, TOWN_RADIUS_KM)
    lat2, lon2, town2 = place_cabinets(n_scattered, SCATTER_RADIUS_KM)
    kind = np.array(["town"] * len(lat1) + ["scattered"] * len(lat2))
    # Shuffle so town and scattered cabinets are mixed in the ID order, like real data
    order = rng.permutation(N_CABINETS)
    df = pd.DataFrame({
        "CabinetId": [f"CAB{i:04d}" for i in range(1, N_CABINETS + 1)],
        "cabinetlat": np.round(np.concatenate([lat1, lat2])[order], 6),
        "cabinetlong": np.round(np.concatenate([lon1, lon2])[order], 6),
        "total_tickets": make_tickets(N_CABINETS),
        "total_days": TOTAL_DAYS,
    })
    df["demand"] = (df["total_tickets"] / df["total_days"]).round(3)
    return df, np.concatenate([town1, town2])[order], kind[order]

## Make the dataset
The summary shows the numbers the clustering cares about. **Possible clusters** is the same range the clustering
notebook prints: from `ceil(total demand / CAP)` (every day full) to `floor(total demand / SMALLEST_MIN)` (every day
one person at the minimum).

In [ ]:
cabinets, home_town, kind = build_dataset()
total = cabinets["demand"].sum()
summary = pd.Series({
    "Cabinets": f"{len(cabinets):,}",
    "Town / scattered": f"{(kind == 'town').sum():,} / {(kind == 'scattered').sum():,}",
    "Total demand (tickets/day)": f"{total:,.1f}",
    "Mean demand per cabinet": f"{cabinets['demand'].mean():.3f}",
    "Busiest cabinet (tickets/day)": f"{cabinets['demand'].max():.2f}",
    "Possible clusters": f"{math.ceil(total / CAP - 1e-9)}–{math.floor(total / SMALLEST_MIN + 1e-9)}",
})
print(summary.to_string())
print("Cabinets around each town (town + scattered):")
print(pd.Series(home_town).value_counts().to_string())
cabinets.head()

## Partitions
Each cabinet gets the partition of its strip: the area is cut from west to east at the longitudes that give every
partition its share of `PARTITION_SHARES`. Then the TRIS cabinets closest to a cut, `SHARED_SHARE` of them, are also
listed under the TROS partition on the other side of that cut, with their own tickets: the same cabinet can be work
for two teams, and the model plans it in each partition separately.

In [ ]:
def add_partitions(df):
    # The cabinet table with a partition column (and the shared cabinets added), or as it is without partitions.
    if not PARTITION_SHARES:
        return df
    names = list(PARTITION_SHARES)
    shares = np.array(list(PARTITION_SHARES.values()), dtype=float)
    cuts = np.quantile(df["cabinetlong"], np.cumsum(shares / shares.sum())[:-1])
    df = df.assign(partition=np.array(names)[np.searchsorted(cuts, df["cabinetlong"], side="right")])
    tris = df[df["partition"].str.startswith("TRIS")]
    gap = np.abs(tris["cabinetlong"].to_numpy()[:, None] - cuts[None, :])     # longitude distance to every cut
    near = gap.argmin(axis=1)
    pick = np.argsort(gap.min(axis=1))[:int(round(SHARED_SHARE * len(tris)))]
    shared = tris.iloc[pick].copy()
    # the partition on the other side of the nearest cut
    side = np.searchsorted(cuts, shared["cabinetlong"], side="right")
    other = np.where(side > near[pick], near[pick], near[pick] + 1)
    shared["partition"] = np.array(names)[other]
    shared["total_tickets"] = make_tickets(len(shared))
    shared["demand"] = (shared["total_tickets"] / shared["total_days"]).round(3)
    return pd.concat([df, shared], ignore_index=True)


cabinets = add_partitions(cabinets)
if "partition" in cabinets:
    per = cabinets.groupby("partition").agg(cabinets=("CabinetId", "size"), demand=("demand", "sum"))
    print(per.round(1).to_string())
    print(f"{int(cabinets['CabinetId'].duplicated().sum())} cabinet(s) are in two partitions")

## Check with the dashboard's rules
This runs the check the dashboard uses when a file is uploaded in **New run**, once per partition (each partition
is planned on its own). Errors mean the file can't be clustered with these rules, for example a cabinet busier than
`CAP`.

In [ ]:
sys.path.insert(0, str(ROOT / "dashboard"))
from results import validate_upload  # noqa: E402

rules = {"PEOPLE_PER_CLUSTER": PEOPLE_PER_CLUSTER, "DEMAND_PER_PERSON": DEMAND_PER_PERSON}
groups = cabinets.groupby("partition") if "partition" in cabinets else [("all cabinets", cabinets)]
ok = True
for name, part in groups:
    clean, errors, warnings = validate_upload(part.drop(columns="partition", errors="ignore"), rules)
    for e in errors:
        print(f"ERROR ({name}):", e)
    for w in warnings:
        print(f"Note ({name}):", w)
    ok &= clean is not None
print("Check passed." if ok else "Check failed: change the settings above.")

## Preview
Each dot is a cabinet; bigger and darker means more tickets per day. Town centres are the labelled pins.
The histogram shows how demand is shared out between cabinets.

In [ ]:
def preview_map(df):
    m = folium.Map(location=CENTRE, zoom_start=10, tiles="cartodbpositron")
    top = df["demand"].quantile(0.95)
    for row in df.itertuples():
        share = min(row.demand / top, 1)
        folium.CircleMarker(
            (row.cabinetlat, row.cabinetlong), radius=3 + 5 * share, weight=0, fill=True, fill_opacity=0.75,
            fill_color="#fb5e00" if share > 0.66 else "#1800e1" if share > 0.2 else "#8c86f0",
            tooltip=f"{row.CabinetId}: {row.demand:.2f} tickets/day").add_to(m)
    for t in towns.itertuples():
        folium.Marker((t.lat, t.long), tooltip=t.town, icon=folium.Icon(color="darkblue", icon="info-sign")).add_to(m)
    half = AREA_KM / 2
    corner1 = offset_km(CENTRE[0], CENTRE[1], -half, -half)
    corner2 = offset_km(CENTRE[0], CENTRE[1], half, half)
    folium.Rectangle([corner1, corner2], color="#150080", weight=1, fill=False, dash_array="6").add_to(m)
    return m


preview_map(cabinets)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(cabinets["demand"], bins=40, color="#1800e1")
ax.set_xlabel("demand (tickets/day)")
ax.set_ylabel("cabinets")
ax.set_title("Demand per cabinet")
plt.show()

## Save
The file goes into `Dataset/`.

In [ ]:
name = OUTPUT_NAME or f"generated_{REGION.lower().replace(' ', '_')}_{N_CABINETS}cab_seed{SEED}.csv"
out_path = DATA_DIR / name
cabinets.to_csv(out_path, index=False)
print(f"Saved {len(cabinets):,} cabinets to {out_path.relative_to(ROOT).as_posix()}")